##### 要用的函数

In [2]:
import torch
from torch.utils.data import DataLoader

class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs): # 将输入和输出的数量编码为变量，使我们可以在具有不同特征数量和类别数量的数据集上重复相同的代码
        super().__init__()

        self.layers = torch.nn.Sequential(
            # 第一个隐藏层
            torch.nn.Linear(num_inputs, 30), # 线性层将输入结点和输出结点的数量作为参数
            torch.nn.ReLU(), # 非线性激活函数被放置在隐藏层之间

            # 第二个隐藏层
            torch.nn.Linear(30, 20), # 下一个隐藏层的输出节点数量必须与下一层的输入节点数量相匹配
            torch.nn.ReLU(),

            # 输出层
            torch.nn.Linear(20, num_outputs)
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits # 最后一层的输出称为 logits

from torch.utils.data import Dataset

import torch

x_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5],
])
y_train = torch.tensor([0, 0, 0, 1, 1])

class ToyDataset(Dataset):
    def __init__(self, x, y):
        self.features = x
        self.labels = y

    # 检索一条数据记录及其对应标签的说明
    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y

    def __len__(self):
        return self.labels.shape[0] # 返回数据集总长度的说明

train_ids = ToyDataset(x_train, y_train)

train_loader = DataLoader(
    dataset=train_ids,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    drop_last=True, # 丢弃最后一个批次的数据
)

### A.7 典型的训练循环
#### 代码清单 A-9 在 PyTorch 中进行神经网络训练

In [4]:
import torch.nn.functional as F

torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2) # 该数据集有两个特征和两个类别
optimizer = torch.optim.SGD(
    model.parameters(), lr = 0.5
) # 优化器需要知道哪些参数需要优化

num_epochs = 3
for epoch in range(num_epochs):

    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):
        logits = model(features)

        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad() # 将上一轮的梯度置0，以防止意外的梯度累积
        loss.backward() # 根据模型参数计算损失的梯度
        optimizer.step() # 优化器使用梯度更新模型参数

        ### LOGGING
        print(f"Epoch {epoch + 1: 03d}/{num_epochs: 03d}"
              f" | Batch {batch_idx: 03d}/{len(train_loader): 03d}"
              f" | Train Loss: {loss:.2f}")

    model.eval()
    # 插入可选的模型评估代码

Epoch  01/ 03 | Batch  00/ 02 | Train Loss: 0.75
Epoch  01/ 03 | Batch  01/ 02 | Train Loss: 0.65
Epoch  02/ 03 | Batch  00/ 02 | Train Loss: 0.44
Epoch  02/ 03 | Batch  01/ 02 | Train Loss: 0.13
Epoch  03/ 03 | Batch  00/ 02 | Train Loss: 0.03
Epoch  03/ 03 | Batch  01/ 02 | Train Loss: 0.00
